In [1]:
from core_entities import *
from sqlalchemy.orm import selectinload, joinedload

engine = redeclare_db()


with Session(engine) as session:
    user1 = User(name = "Alice", posts = [Post(text="Alicetext1"),Post(text="Alicetext2")])
    user2 = User(name = "Bob", posts = [Post(text="Alicetext2")])
    user3 = User(name = "Charlie", posts = [])
    session.add_all([user1, user2])
    session.commit()

2026-09-10 10:01:52,694 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-10 10:01:52,695 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-09-10 10:01:52,696 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-10 10:01:52,697 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("users")
2026-09-10 10:01:52,697 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-10 10:01:52,698 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("posts")
2026-09-10 10:01:52,699 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-10 10:01:52,700 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("posts")
2026-09-10 10:01:52,700 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-10 10:01:52,702 INFO sqlalchemy.engine.Engine 
CREATE TABLE users (
	name VARCHAR NOT NULL, 
	nickname VARCHAR, 
	id INTEGER NOT NULL, 
	PRIMARY KEY (id)
)


2026-09-10 10:01:52,703 INFO sqlalchemy.engine.Engine [no key 0.00090s] ()
2026-09-10 10:01:52,705 INFO sqlalchemy.engine.Engine 
CREATE TABLE posts (


In [12]:
with Session(engine) as session:

    user3 = User(name = "Charlie", posts = [])
    session.add_all([user3])
    session.commit()

2026-09-10 10:17:34,207 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-10 10:17:34,208 INFO sqlalchemy.engine.Engine INSERT INTO users (name, nickname) VALUES (?, ?)
2026-09-10 10:17:34,209 INFO sqlalchemy.engine.Engine [generated in 0.00078s] ('Charlie', None)
2026-09-10 10:17:34,210 INFO sqlalchemy.engine.Engine COMMIT


In [5]:
#выбрать пользователей, у которых есть хотя бы один запрос
with Session(engine) as session:
    sbquery = select(Post.user_id).where(Post.text.startswith("Alice"))
    stmt = select(User.name, User.nickname).where(User.id.in_(sbquery))
    result = session.execute(stmt)
    for data in result.scalars():
        print(data)

2026-09-10 10:06:05,702 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-10 10:06:05,704 INFO sqlalchemy.engine.Engine SELECT users.name, users.nickname 
FROM users 
WHERE users.id IN (SELECT posts.user_id 
FROM posts 
WHERE (posts.text LIKE ? || '%'))
2026-09-10 10:06:05,705 INFO sqlalchemy.engine.Engine [cached since 11s ago] ('Alice',)
Alice
Bob
2026-09-10 10:06:05,706 INFO sqlalchemy.engine.Engine ROLLBACK


In [16]:
#DISTINCT VS UNIQUE
#DISTINCT - sql-side
#UNIQUE - alchemy-side
with Session(engine) as session:
    stmt = select(User.name).outerjoin(User.posts)
    result = session.execute(stmt)
    for data in result.unique():
        print(data)

with Session(engine) as session:
    stmt = select(User.name).outerjoin(User.posts).distinct()
    result = session.execute(stmt)
    for data in result:
        print(data)

2026-09-10 10:18:19,341 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-10 10:18:19,342 INFO sqlalchemy.engine.Engine SELECT users.name 
FROM users LEFT OUTER JOIN posts ON users.id = posts.user_id
2026-09-10 10:18:19,343 INFO sqlalchemy.engine.Engine [cached since 115.1s ago] ()
('Alice',)
('Bob',)
('Charlie',)
2026-09-10 10:18:19,345 INFO sqlalchemy.engine.Engine ROLLBACK
2026-09-10 10:18:19,346 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-10 10:18:19,347 INFO sqlalchemy.engine.Engine SELECT DISTINCT users.name 
FROM users LEFT OUTER JOIN posts ON users.id = posts.user_id
2026-09-10 10:18:19,348 INFO sqlalchemy.engine.Engine [cached since 9.296s ago] ()
('Alice',)
('Bob',)
('Charlie',)
2026-09-10 10:18:19,351 INFO sqlalchemy.engine.Engine ROLLBACK
